# Sentiment Analysis - Movies Review Evaluator 

## Libraries and imports:

In [1]:
import pandas as pd 
import nltk
import string
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

## Data Collection and Loading: 

In [2]:
# IMBD dataset with 50k reviews (it can be found following the source below): 
# https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews?resource=download
df = pd.read_csv('../IMDB Dataset.csv')

## Exploratory Data Analysis (EDA)

In [3]:
# Access to head (first 5 elements) of the csv file
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
# Using shape to understand the amount of reviews and columns in the file
print (df.shape)

(50000, 2)


In [5]:
# Using info for more detailled information on the dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


In [6]:
# Considering 4 out of 5 reviews are rated 'positive' in head(), checking for balance in 'sentiment' column:
print(df['sentiment'].value_counts())

sentiment
positive    25000
negative    25000
Name: count, dtype: int64


## Data Preprocessing

In [7]:
# Data cleansing (removing stopwords, uppercases, punctuation)
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/yaradaudt/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [8]:
# Function that removes punctuation, converts uppercase into lowercase and removes the stop words.
def cleansed_text(text):
    """ Removes punctuation, converts to lower case and removes stopwords. """
    text_without_punctuation = "".join([char for char in text if char not in string.punctuation])
    clean_text = text_without_punctuation.lower()
    stop_words = stopwords.words('english')
    expressions = clean_text.split()
    final_text = [expression for expression in expressions if expressions not in stop_words]

    return " ".join(final_text)

In [9]:
# Applying the function in the 'review' column in dataframe, to deliver the (now cleaned) review content.
df['clean_review'] = df['review'].apply(cleansed_text)

In [10]:
# Checking how the "clean_review" content is looking like:
df.head()

,review,sentiment,clean_review
0,One of the other reviewers has mentioned that ...,positive,one of the other reviewers has mentioned that ...
1,A wonderful little production. <br /><br />The...,positive,a wonderful little production br br the filmin...
2,I thought this was a wonderful way to spend ti...,positive,i thought this was a wonderful way to spend ti...
3,Basically there's a family where a little boy ...,negative,basically theres a family where a little boy j...
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,petter matteis love in the time of money is a ...


## Feature Engineering: Text Vectorization

In [11]:
# Instances vectorizer - words will be vectorized into numbers for model understanding
vectorizer = TfidfVectorizer()

In [12]:
# Transforming X vector 'clean_review'
X = vectorizer.fit_transform(df['clean_review'])

In [13]:
# Prints the shape of the X vector 'clean_review' (the amount of unique words in its content) 
print(X.shape)

(50000, 180395)


In [14]:
# Transforming y vector, labeled 'sentiment'
# Instances the label encoder
le = LabelEncoder()
# Transforms values 'positive' and 'negative' in 'sentiment' into 0 and 1 (binary)
y = le.fit_transform(df['sentiment'])

In [15]:
print(y)

[1 1 1 ... 0 0 0]


## Model Training: Scikit-learn (Machine Learning)

In [16]:
# Splitting data into training and test, with test size in a range of 20%.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [17]:
# Logistic Regression
model = LogisticRegression()
model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


## Model Evaluation: Scikit-learn

In [18]:
# Predictions and Accuracy
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f'Model accuracy: {accuracy:.2f}')

Model accuracy: 0.90


In [19]:
# Classification Report 
print('\nClassification Report:\n')
print(classification_report(y_test, y_pred))


Classification Report:

              precision    recall  f1-score   support

           0       0.91      0.89      0.90      4961
           1       0.89      0.91      0.90      5039

    accuracy                           0.90     10000
   macro avg       0.90      0.90      0.90     10000
weighted avg       0.90      0.90      0.90     10000



In [20]:
# Confusion Matrix
matrix_confusion = confusion_matrix(y_test, y_pred)
print('Confusion Matrix:\n')
print(matrix_confusion)

Confusion Matrix:

[[4403  558]
 [ 439 4600]]


## Real-Time Prediction Example: Scikit-learn

In [21]:
new_review = "this movie was actually insanely bad! the plot is weak, and the actors are not really convincing."
cleansed_new_review = cleansed_text(new_review)
vectorized_new_review = vectorizer.transform([cleansed_new_review])
prediction = model.predict(vectorized_new_review)
if prediction[0] == 1:
    print ("The movie is rated as good! (positive review)")
else:
    print ("The movie is rated as bad! (negative review)")

The movie is rated as bad! (negative review)


## Model Training: Pytorch (Deep Learning)

In [22]:
# Splitting data into training and test, with test size in a range of 20%.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [23]:
# Class that teaches Pytorch to deal with its matrix, converting data one by one upon request.
class SparseDataset(Dataset):
    def __init__(self, X_sparse, y):
        self.X_sparse = X_sparse
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return self.X_sparse.shape[0]

    def __getitem__(self, idx):
        x_dense = torch.tensor(self.X_sparse[idx].toarray().squeeze(0), dtype=torch.float32)
        return x_dense, self.y[idx].unsqueeze(0)
        
train_dataset = SparseDataset(X_train, y_train)
test_dataset = SparseDataset(X_test, y_test)

In [24]:
# To prevent issues with memory, data loader loads small batches of data for a better performance
# processing the dataset
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [25]:
# Neural Network class
class Net(nn.Module):
    def __init__(self, n_features):
        super(Net, self).__init__()
        # Model entrance, receiving the 180.395 attributes
        self.fc1 = nn.Linear(n_features, 100)
        # Adding non-linearity 
        self.relu = nn.ReLU()
        # Output layer (produces the models prediction)
        self.fc2 = nn.Linear(100, 1)
    # Defines the processing of the dataset through the neural network
    def forward(self, x):
        return torch.sigmoid(self.fc2(self.relu(self.fc1(x))))

n_features = X_train.shape[1]
# Creates the model
model_pytorch = Net(n_features)

In [26]:
# Loss function to stablish how "wrong" the model can be in its predictions
criterion = nn.BCELoss()
# Optmize the accuracy by adjusting the parameters of the model
optimizer = optim.Adam(model_pytorch.parameters(), lr=0.001)

In [27]:
# Amount of epochs (rounds) in which the model will pass through the dataset entirely
epochs = 10

# Training Loop
for epoch in range(epochs):
    for inputs, labels in train_loader:
        # Training steps
        optimizer.zero_grad()
        outputs = model_pytorch(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
    # Prints each epoch and loss values
    print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

Epoch [1/10], Loss: 0.2121
Epoch [2/10], Loss: 0.0868
Epoch [3/10], Loss: 0.0396
Epoch [4/10], Loss: 0.0135
Epoch [5/10], Loss: 0.0038
Epoch [6/10], Loss: 0.0030
Epoch [7/10], Loss: 0.0014
Epoch [8/10], Loss: 0.0003
Epoch [9/10], Loss: 0.0007
Epoch [10/10], Loss: 0.0005


## Model Evaluation: Pytorch

In [28]:
model_pytorch.eval()

Net(
  (fc1): Linear(in_features=180395, out_features=100, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=100, out_features=1, bias=True)
)

In [29]:
# List of predictions (saved as an array)
y_pred_list = []
# Disables gradient calculations (no longer necessary, training is complete)
with torch.no_grad(): 
    # Iteration loop (reviews and labels (positive negative))
    for inputs, labels in test_loader:
        # Executes prediction
        outputs = model_pytorch(inputs)
        # Returns the probabilities, and rounds the numbers approximately (closer to 0 or 1 to define positive/negative espectrum)
        y_pred_tag = torch.round(outputs)
        # Converts the Pytorch tensor into a Numpy array, and extends the former y_pred_list with this array
        y_pred_list.extend(y_pred_tag.cpu().numpy())

In [30]:
# Accuracy
accuracy = accuracy_score(y_test, y_pred_list)
print(f'Accuracy of PyTorch Model: {accuracy:.2f}')

Accuracy of PyTorch Model: 0.89


In [31]:
# Classification Report
print('\nPyTorch Classification Report:\n')
print(classification_report(y_test, y_pred_list))


PyTorch Classification Report:

              precision    recall  f1-score   support

           0       0.90      0.88      0.89      4961
           1       0.89      0.90      0.89      5039

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000



In [32]:
# Confusion Matrix
print('\nPytorch Confusion Matrix:\n')
print(confusion_matrix(y_test, y_pred_list))


Pytorch Confusion Matrix:

[[4382  579]
 [ 502 4537]]


## Real-Time Prediction Example: Pytorch

In [33]:
newer_review = "This movie was terribly amazing! It was scary, but so good!"
cleansed_newer_review = cleansed_text(newer_review)

vectorized_newer_review = vectorizer.transform([cleansed_newer_review])

tensor_review = torch.from_numpy(vectorized_newer_review.toarray()).float()

model_pytorch.eval()

with torch.no_grad():
    outputs = model_pytorch(tensor_review)
    prediction = torch.round(outputs)

if prediction.item() == 1:
    print("This movie is rated as good! (positive review)")
else:
    print("This movie is rated as bad! (negative review)")

This movie is rated as good! (positive review)
